In [0]:
# ============================================================
# 03_SILVER_SCD2
# Purpose : Apply SCD Type 2 logic to slowly changing
#           dimensions — vessels, ports, accounts
#
# Input  : silver/xxx_all_versions  (cleaned, all versions)
# Output : silver/dim_xxx           (SCD2 historised dimension)
#
# SCD2 columns added:
#   valid_from   DATE     — when this version became active
#   valid_to     DATE     — when this version was superseded
#                           NULL = currently active
#   is_current   BOOLEAN  — True for the current version only
#   row_hash     STRING   — MD5 of tracked columns (change detection)
#   surrogate_sk BIGINT   — warehouse-generated unique key
# ============================================================

MOUNT       = "/mnt/commercialbdi_lob_commercialbdi_dev/rocky/shipping_project"
BRONZE_PATH  = f"{MOUNT}/bronze"
SILVER_PATH = f"{MOUNT}/silver"

print(f"Silver : {SILVER_PATH}")

Silver : /mnt/commercialbdi_lob_commercialbdi_dev/rocky/shipping_project/silver


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable
import pyspark.sql.types as T

def compute_hash(df, tracked_cols):
    """
    Computes MD5 hash of all tracked columns.
    Uses || separator to prevent false hash collisions.
    NULLs are coalesced to empty string before hashing
    so NULL == NULL doesn't trigger a false change.
    """
    return df.withColumn("row_hash",
        F.md5(
            F.concat_ws("||",
                *[F.coalesce(F.col(c).cast("string"), F.lit(""))
                  for c in tracked_cols]
            )
        )
    )

print("✓ Helper functions ready")

✓ Helper functions ready


In [0]:
# where change and main file reside - 40 rows found.
spark.read.format("delta").load(f"{BRONZE_PATH}/accounts").display()

account_id,account_name,industry,account_tier,region,country_code,annual_revenue_usd,employee_count,primary_trade_lane,owner_user_id,is_active,created_date,last_modified_date,_extracted_at,_bronze_ingested_at,_source_object,_source_path,_bronze_year,_bronze_month
ACC2D991B7A868C,"Beard, Peters and Black",Automotive,Silver,Latin America,CO,488699683,17458,JP→NL,USR9FE914DC2800,True,2022-04-26,2024-07-15,2026-06-29T10:17:38Z,2026-06-30T12:25:46.168Z,accounts,dbfs:/mnt/commercialbdi_lob_commercialbdi_dev/rocky/shipping_project/landing/accounts/accounts_full.csv,2026,6
ACCE074D97F96C9,Freeman-Chang,Pharmaceuticals,Silver,Latin America,PE,395349899,21173,IN→DE,USRF54A84DD8911,True,2022-01-21,2022-03-04,2026-06-29T10:17:38Z,2026-06-30T12:25:46.168Z,accounts,dbfs:/mnt/commercialbdi_lob_commercialbdi_dev/rocky/shipping_project/landing/accounts/accounts_full.csv,2026,6
ACC88AF6AD82FD6,King-Martinez,Retail,Gold,Latin America,CO,297487718,26788,SG→FR,USRF4EF15C034D8,False,2022-10-10,2023-02-03,2026-06-29T10:17:38Z,2026-06-30T12:25:46.168Z,accounts,dbfs:/mnt/commercialbdi_lob_commercialbdi_dev/rocky/shipping_project/landing/accounts/accounts_full.csv,2026,6
ACCCC37B5DB9B39,"Washington, Hardy and Bray",FMCG,Gold,Latin America,CL,465533246,9788,MY→PL,USR19509359C4A7,True,2022-06-02,2023-12-19,2026-06-29T10:17:38Z,2026-06-30T12:25:46.168Z,accounts,dbfs:/mnt/commercialbdi_lob_commercialbdi_dev/rocky/shipping_project/landing/accounts/accounts_full.csv,2026,6
ACCFE31DF3E6E95,"Carlson, Hooper and Wall",Chemicals,Silver,Latin America,PE,307981523,25148,KR→SE,USR3A13C5891785,True,2022-04-07,2024-05-08,2026-06-29T10:17:38Z,2026-06-30T12:25:46.168Z,accounts,dbfs:/mnt/commercialbdi_lob_commercialbdi_dev/rocky/shipping_project/landing/accounts/accounts_full.csv,2026,6
ACCFCE5ADF8E73A,Wilson-Jones,Agriculture,Silver,Latin America,CL,246261856,19849,KR→SE,USR62866EA41119,True,2022-12-20,2023-04-08,2026-06-29T10:17:38Z,2026-06-30T12:25:46.168Z,accounts,dbfs:/mnt/commercialbdi_lob_commercialbdi_dev/rocky/shipping_project/landing/accounts/accounts_full.csv,2026,6
ACC4289E540EC98,"Duran, Obrien and Gibbs",Technology,Gold,Latin America,CL,402300732,27280,JP→FR,USR6AA3DB66E051,True,2022-06-01,2023-12-05,2026-06-29T10:17:38Z,2026-06-30T12:25:46.168Z,accounts,dbfs:/mnt/commercialbdi_lob_commercialbdi_dev/rocky/shipping_project/landing/accounts/accounts_full.csv,2026,6
ACC389A2C5A956A,Arnold and Sons,FMCG,Gold,Latin America,CO,331525766,26796,IN→DE,USR54E7D19D5D69,True,2022-02-20,2023-05-10,2026-06-29T10:17:38Z,2026-06-30T12:25:46.168Z,accounts,dbfs:/mnt/commercialbdi_lob_commercialbdi_dev/rocky/shipping_project/landing/accounts/accounts_full.csv,2026,6
ACCCE4444B7E6B8,"Walker, Hernandez and Baker",Retail,Silver,Latin America,BR,391435609,43231,IN→PL,USR122BEA281760,True,2022-11-11,2022-12-12,2026-06-29T10:17:38Z,2026-06-30T12:25:46.168Z,accounts,dbfs:/mnt/commercialbdi_lob_commercialbdi_dev/rocky/shipping_project/landing/accounts/accounts_full.csv,2026,6
ACCFB5ADC4A3E61,Larson-Holloway,Technology,Bronze,Latin America,CO,205496488,39200,IN→PL,USR6EEE6E5AAA58,True,2022-11-16,2024-01-30,2026-06-29T10:17:38Z,2026-06-30T12:25:46.168Z,accounts,dbfs:/mnt/commercialbdi_lob_commercialbdi_dev/rocky/shipping_project/landing/accounts/accounts_full.csv,2026,6


In [0]:
# where change and main file reside - 80 rows found.(duplicate values) we are implementing SCD-2 with row_hashing
spark.read.format("delta").load(f"{SILVER_PATH}/accounts_all_versions").filter(F.col("account_id").isin("ACC64441431ABCC", "ACC59C2C75D1233", "ACCDCF205F2DAD8", "ACCBC8556543E1F", "ACCDAA3A2C37A28", "ACCA012470B509D", "ACCD933A1F8CCEB", "ACC938B9DB0F149", "ACCA220763123BB", "ACC7CC70F650B76", "ACC33F434AE016E", "ACC05B7DE71153E", "ACC7A4633D6DE7D", "ACC4FBDAA4A22BB", "ACCECF12BF6240F", "ACC78F4BE89B0D4", "ACC6B349845B905", "ACC19D4A5307B33", "ACCACEA24E87B09", "ACC2B78FB82CAE4", "ACC2107FA6AEFBD", "ACC54E38CF61EAC", "ACC1BE4DC5D4AFA", "ACCF93BDBBBE03A", "ACCDF4917DB61DC", "ACCD1BCABC50681", "ACC308D9E21CFFF", "ACC9B4C21FF293F", "ACC01594092CCEF", "ACC3360FA849C3A", "ACCA29469208805", "ACC6D4A914DE443", "ACCED366DD57499", "ACC6D36C9F8E67B", "ACC5CFCFB1735C8", "ACC084D36202D5C", "ACC1BE271495132", "ACC09E0DE9CFB47", "ACC85ACB6027ACC", "ACC0C068320F0B8")).orderBy(F.col("account_id").desc()).display()

account_id,account_name,industry,account_tier,region,country_code,annual_revenue_usd,employee_count,primary_trade_lane,owner_user_id,is_active,created_date,last_modified_date,_extracted_at,_source_object,_silver_cleaned_at
ACCF93BDBBBE03A,Young-Allen,Fmcg,Bronze,Europe,PL,408749856,33899,AU→DE,USREA4E5D3E3017,true,2022-07-28,2023-01-06,2026-06-29T10:17:38Z,accounts,2026-07-01T08:28:29.342Z
ACCF93BDBBBE03A,Young-Allen,Fmcg,Bronze,North America,CA,408749856,33899,AU→DE,USREA4E5D3E3017,true,2022-07-28,2024-10-14,2024-10-14T03:00:00Z,accounts,2026-07-01T08:28:29.342Z
ACCED366DD57499,Hudson-Barnett,Pharmaceuticals,Platinum,Latin America,CL,326514309,39117,KR→SE,USR11D5E158774B,true,2022-10-13,2024-03-07,2024-03-07T03:00:00Z,accounts,2026-07-01T08:28:29.342Z
ACCED366DD57499,Hudson-Barnett,Pharmaceuticals,Platinum,North America,US,326514309,39117,KR→SE,USR11D5E158774B,true,2022-10-13,2024-01-09,2026-06-29T10:17:38Z,accounts,2026-07-01T08:28:29.342Z
ACCECF12BF6240F,Wolf-Harris,Manufacturing,Bronze,Middle East,KW,347102674,5967,TH→PL,USRFE09863590A8,true,2022-05-27,2024-08-14,2026-06-29T10:17:38Z,accounts,2026-07-01T08:28:29.342Z
ACCECF12BF6240F,Wolf-Harris,Manufacturing,Silver,Middle East,KW,347102674,5967,TH→PL,USRFE09863590A8,true,2022-05-27,2023-09-22,2023-09-22T03:00:00Z,accounts,2026-07-01T08:28:29.342Z
ACCDF4917DB61DC,Jones Inc,Retail,Bronze,Middle East,QA,270563672,37485,JP→NL,USR042559435FDC,false,2023-01-01,2024-04-04,2026-06-29T10:17:38Z,accounts,2026-07-01T08:28:29.342Z
ACCDF4917DB61DC,Jones Inc,Retail,Bronze,Europe,NO,270563672,37485,JP→NL,USR042559435FDC,false,2023-01-01,2024-12-24,2024-12-24T03:00:00Z,accounts,2026-07-01T08:28:29.342Z
ACCDCF205F2DAD8,"Graham, Olsen and Costa",Fmcg,Bronze,Africa,NG,283895594,43771,IN→DE,USR11D5E158774B,true,2022-05-26,2024-07-13,2024-07-13T03:00:00Z,accounts,2026-07-01T08:28:29.342Z
ACCDCF205F2DAD8,"Graham, Olsen and Costa",Fmcg,Bronze,Latin America,BR,283895594,43771,IN→DE,USR11D5E158774B,true,2022-05-26,2024-01-29,2026-06-29T10:17:38Z,accounts,2026-07-01T08:28:29.342Z


In [0]:
# ── ACCOUNTS SCD2 ─────────────────────────────────────────────
#
# Tracked columns (changes to these create a new SCD2 row):
#   account_tier  → tier upgrade (Bronze→Silver→Gold) is a business event
#   region        → region reassignment affects reporting
#   country_code  → country change affects geographic analysis
#   is_active     → activation/deactivation is meaningful history
#
# NOT tracked (intentionally excluded from hash):
#   account_name      → name corrections are Type 1 (just update)
#   annual_revenue_usd → financial estimate, not a stable tracked attribute
#   last_modified_date → Salesforce metadata, changes on every edit
# ─────────────────────────────────────────────────────────────

ACCOUNTS_TRACKED = ["account_tier", "region", "country_code", "is_active"]
DIM_ACCOUNTS_PATH = f"{SILVER_PATH}/dim_accounts"

# ── Step 1: Load all versions from silver cleaning output ─────
df_all = spark.read.format("delta").load(f"{SILVER_PATH}/accounts_all_versions")

print(f"Total rows (full + changes): {df_all.count():,}")
print(f"Distinct account_ids       : {df_all.select('account_id').distinct().count():,}")

# ── Step 2: Compute row hash on all versions ──────────────────
df_hashed = compute_hash(df_all, ACCOUNTS_TRACKED)

# ── Step 3: Reconstruct history per account ───────────────────
# Each account may have multiple rows (original + 1 change)
# We need to order them by last_modified_date to know which came first
# Then assign valid_from / valid_to per version

# Window: for each account, order versions by when they were modified
w_account = Window.partitionBy("account_id").orderBy("last_modified_date")

df_versioned = (df_hashed

    # Assign version number within each account
    .withColumn("version_number",
        F.row_number().over(w_account))

    # valid_from = the date this version was modified/created
    .withColumn("valid_from",
        F.col("last_modified_date"))

    # valid_to = the day before the NEXT version's valid_from
    # For the latest version: valid_to = NULL (still current)
    .withColumn("valid_to",
        F.date_sub(
            F.lead("last_modified_date", 1)
             .over(w_account),
            1)
    )

    # is_current = True only for the latest version per account
    .withColumn("is_current",
        F.lead("last_modified_date", 1).over(w_account).isNull())

    # Surrogate key = hash of account_id + version for uniqueness
    # In production you'd use a sequence/identity column
    # Here we use a monotonically increasing ID assigned after ordering
)

# Assign surrogate key — unique integer per row
df_versioned = df_versioned.withColumn(
    "account_sk",
    F.monotonically_increasing_id()
)

# ── Step 4: Select final dimension columns ────────────────────
df_dim_accounts = df_versioned.select(
    # Surrogate key (warehouse-generated — used in fact table FKs)
    F.col("account_sk"),

    # Natural key (from source system — never used as FK)
    F.col("account_id"),

    # Tracked attributes (the ones in our hash)
    F.col("account_tier"),
    F.col("region"),
    F.col("country_code"),
    F.col("is_active"),

    # Non-tracked attributes (Type 1 — just take latest value)
    F.col("account_name"),
    F.col("industry"),
    F.col("annual_revenue_usd"),
    F.col("employee_count"),
    F.col("primary_trade_lane"),
    F.col("owner_user_id"),

    # SCD2 control columns
    F.col("valid_from"),
    F.col("valid_to"),
    F.col("is_current"),
    F.col("row_hash"),
    F.col("version_number"),
    F.col("_silver_cleaned_at")
)

# ── Step 5: Write dim_accounts ────────────────────────────────
(df_dim_accounts.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(DIM_ACCOUNTS_PATH))

# ── Step 6: Verify the SCD2 output ───────────────────────────
df_verify = spark.read.format("delta").load(DIM_ACCOUNTS_PATH)

total_rows    = df_verify.count()
current_rows  = df_verify.filter(F.col("is_current") == True).count()
history_rows  = df_verify.filter(F.col("is_current") == False).count()
accounts_with_history = df_verify.groupBy("account_id") \
    .count().filter(F.col("count") > 1).count()

print(f"\ndim_accounts SCD2 results:")
print(f"  Total rows              : {total_rows:,}")
print(f"  Current rows (is_current=True)  : {current_rows:,}")
print(f"  Historical rows (is_current=False): {history_rows:,}")
print(f"  Accounts with > 1 version       : {accounts_with_history:,}")
print(f"\n  ✓ Each account_id has exactly one is_current=True row:")

# Integrity check — if any account has 2+ current rows, SCD2 is broken
duplicates = (df_verify
    .filter(F.col("is_current") == True)
    .groupBy("account_id")
    .count()
    .filter(F.col("count") > 1)
    .count())

if duplicates == 0:
    print(f"    ✓ No duplicate current rows — SCD2 integrity confirmed")
else:
    print(f"    ✗ {duplicates} accounts have duplicate current rows — INVESTIGATE")

Total rows (full + changes): 340
Distinct account_ids       : 300

dim_accounts SCD2 results:
  Total rows              : 340
  Current rows (is_current=True)  : 300
  Historical rows (is_current=False): 40
  Accounts with > 1 version       : 40

  ✓ Each account_id has exactly one is_current=True row:
    ✓ No duplicate current rows — SCD2 integrity confirmed


In [0]:
df_dim_accounts.filter(F.col("account_id").isin("ACC64441431ABCC", "ACC59C2C75D1233", "ACCDCF205F2DAD8", "ACCBC8556543E1F", "ACCDAA3A2C37A28", "ACCA012470B509D", "ACCD933A1F8CCEB", "ACC938B9DB0F149", "ACCA220763123BB", "ACC7CC70F650B76", "ACC33F434AE016E", "ACC05B7DE71153E", "ACC7A4633D6DE7D", "ACC4FBDAA4A22BB", "ACCECF12BF6240F", "ACC78F4BE89B0D4", "ACC6B349845B905", "ACC19D4A5307B33", "ACCACEA24E87B09", "ACC2B78FB82CAE4", "ACC2107FA6AEFBD", "ACC54E38CF61EAC", "ACC1BE4DC5D4AFA", "ACCF93BDBBBE03A", "ACCDF4917DB61DC", "ACCD1BCABC50681", "ACC308D9E21CFFF", "ACC9B4C21FF293F", "ACC01594092CCEF", "ACC3360FA849C3A", "ACCA29469208805", "ACC6D4A914DE443", "ACCED366DD57499", "ACC6D36C9F8E67B", "ACC5CFCFB1735C8", "ACC084D36202D5C", "ACC1BE271495132", "ACC09E0DE9CFB47", "ACC85ACB6027ACC", "ACC0C068320F0B8")).orderBy(F.col("account_id").desc()).display()

account_sk,account_id,account_tier,region,country_code,is_active,account_name,industry,annual_revenue_usd,employee_count,primary_trade_lane,owner_user_id,valid_from,valid_to,is_current,row_hash,version_number,_silver_cleaned_at
78,ACCF93BDBBBE03A,Bronze,Europe,PL,true,Young-Allen,Fmcg,408749856,33899,AU→DE,USREA4E5D3E3017,2023-01-06,2024-10-13,false,0d4c37816a627b7a0476ff3ef475eeef,1,2026-07-01T08:28:29.342Z
79,ACCF93BDBBBE03A,Bronze,North America,CA,true,Young-Allen,Fmcg,408749856,33899,AU→DE,USREA4E5D3E3017,2024-10-14,null,true,f9f839d7016fac9a5c92ab81ef6f0e99,2,2026-07-01T08:28:29.342Z
76,ACCED366DD57499,Platinum,North America,US,true,Hudson-Barnett,Pharmaceuticals,326514309,39117,KR→SE,USR11D5E158774B,2024-01-09,2024-03-06,false,3418a27d7287e503e361d5f8a45a2c4e,1,2026-07-01T08:28:29.342Z
77,ACCED366DD57499,Platinum,Latin America,CL,true,Hudson-Barnett,Pharmaceuticals,326514309,39117,KR→SE,USR11D5E158774B,2024-03-07,null,true,713acd20761b9f0bd057eafaa8c41c01,2,2026-07-01T08:28:29.342Z
74,ACCECF12BF6240F,Silver,Middle East,KW,true,Wolf-Harris,Manufacturing,347102674,5967,TH→PL,USRFE09863590A8,2023-09-22,2024-08-13,false,f460ca1dd35b57b0f353b3845034f0e1,1,2026-07-01T08:28:29.342Z
75,ACCECF12BF6240F,Bronze,Middle East,KW,true,Wolf-Harris,Manufacturing,347102674,5967,TH→PL,USRFE09863590A8,2024-08-14,null,true,81818d42f2c8709771f7ce65b4712bfc,2,2026-07-01T08:28:29.342Z
72,ACCDF4917DB61DC,Bronze,Middle East,QA,false,Jones Inc,Retail,270563672,37485,JP→NL,USR042559435FDC,2024-04-04,2024-12-23,false,eb99c0a8d3d4dd2c8f74d514f430a9ce,1,2026-07-01T08:28:29.342Z
73,ACCDF4917DB61DC,Bronze,Europe,NO,false,Jones Inc,Retail,270563672,37485,JP→NL,USR042559435FDC,2024-12-24,null,true,7f3395b7703d2474b854129c6439e73a,2,2026-07-01T08:28:29.342Z
70,ACCDCF205F2DAD8,Bronze,Latin America,BR,true,"Graham, Olsen and Costa",Fmcg,283895594,43771,IN→DE,USR11D5E158774B,2024-01-29,2024-07-12,false,0b810755f7618933fbdf043845cd55d4,1,2026-07-01T08:28:29.342Z
71,ACCDCF205F2DAD8,Bronze,Africa,NG,true,"Graham, Olsen and Costa",Fmcg,283895594,43771,IN→DE,USR11D5E158774B,2024-07-13,null,true,45a0ba3c59e602470819cdeab12bd067,2,2026-07-01T08:28:29.342Z


In [0]:
# ── VESSELS SCD2 ──────────────────────────────────────────────
#
# Tracked columns:
#   flag_country → vessel re-flagging is a legal/compliance event
#   operator     → operator change affects service attribution
#   is_active    → vessel retirement is meaningful history
#
# NOT tracked:
#   vessel_name  → naming corrections are Type 1
#   capacity_teu → physical property, doesn't change
# ─────────────────────────────────────────────────────────────

VESSELS_TRACKED  = ["flag_country", "operator", "is_active"]
DIM_VESSELS_PATH = f"{SILVER_PATH}/dim_vessels"

df_vessels_all = spark.read.format("delta") \
    .load(f"{SILVER_PATH}/vessels_all_versions")

df_vessels_hashed = compute_hash(df_vessels_all, VESSELS_TRACKED)

w_vessel = Window.partitionBy("vessel_id").orderBy("last_updated")

df_dim_vessels = (df_vessels_hashed
    .withColumn("version_number",
        F.row_number().over(w_vessel))
    .withColumn("valid_from",
        F.col("last_updated"))
    .withColumn("valid_to",
        F.date_sub(
            F.lead("last_updated", 1).over(w_vessel), 1))
    .withColumn("is_current",
        F.lead("last_updated", 1).over(w_vessel).isNull())
    .withColumn("vessel_sk",
        F.monotonically_increasing_id())
    .select(
        "vessel_sk", "vessel_id", "imo_number", "vessel_name",
        "vessel_type", "flag_country", "operator", "capacity_teu",
        "year_built", "is_active", "home_port_code",
        "valid_from", "valid_to", "is_current",
        "row_hash", "version_number", "_silver_cleaned_at"
    )
)

(df_dim_vessels.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(DIM_VESSELS_PATH))

total    = df_dim_vessels.count()
current  = df_dim_vessels.filter(F.col("is_current") == True).count()
history  = df_dim_vessels.filter(F.col("is_current") == False).count()

print(f"dim_vessels: {total:,} rows  "
      f"({current:,} current, {history:,} historical)")

dim_vessels: 48 rows  (40 current, 8 historical)


In [0]:
# ── PORTS SCD2 ────────────────────────────────────────────────
#
# Tracked columns:
#   port_type             → reclassification affects routing analysis
#   annual_capacity_teu   → capacity expansion is a notable event
#   is_active             → port closures matter historically
# ─────────────────────────────────────────────────────────────

PORTS_TRACKED  = ["port_type", "annual_capacity_teu", "is_active"]
DIM_PORTS_PATH = f"{SILVER_PATH}/dim_ports"

df_ports_all = spark.read.format("delta") \
    .load(f"{SILVER_PATH}/ports_all_versions")

df_ports_hashed = compute_hash(df_ports_all, PORTS_TRACKED)

w_port = Window.partitionBy("port_id").orderBy("last_updated")

df_dim_ports = (df_ports_hashed
    .withColumn("version_number",
        F.row_number().over(w_port))
    .withColumn("valid_from",
        F.col("last_updated"))
    .withColumn("valid_to",
        F.date_sub(
            F.lead("last_updated", 1).over(w_port), 1))
    .withColumn("is_current",
        F.lead("last_updated", 1).over(w_port).isNull())
    .withColumn("port_sk",
        F.monotonically_increasing_id())
    .select(
        "port_sk", "port_id", "port_code", "port_name",
        "country_code", "region", "port_type",
        "annual_capacity_teu", "is_active",
        "valid_from", "valid_to", "is_current",
        "row_hash", "version_number", "_silver_cleaned_at"
    )
)

(df_dim_ports.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(DIM_PORTS_PATH))

total   = df_dim_ports.count()
current = df_dim_ports.filter(F.col("is_current") == True).count()
history = df_dim_ports.filter(F.col("is_current") == False).count()

print(f"dim_ports: {total:,} rows  "
      f"({current:,} current, {history:,} historical)")

dim_ports: 18 rows  (15 current, 3 historical)


In [0]:
# ── Prove SCD2 works — point-in-time correctness ─────────────
# Pick an account that has 2 versions (upgraded tier)
# Show that querying at different dates returns different results

print("POINT-IN-TIME CORRECTNESS PROOF")
print("="*55)

df_dim = spark.read.format("delta").load(DIM_ACCOUNTS_PATH)

# Find an account that has history (2+ versions)
accounts_with_change = (df_dim
    .groupBy("account_id", "account_name")
    .agg(F.count("*").alias("versions"))
    .filter(F.col("versions") > 1)
    .orderBy("account_id")
)

print("\nAccounts with SCD2 history (sample):")
display(accounts_with_change.limit(5))

# Pick first account with changes and show its full history
sample_id = accounts_with_change.first()["account_id"]

print(f"\nFull SCD2 history for account: {sample_id}")
display(
    df_dim
    .filter(F.col("account_id") == sample_id)
    .select("account_sk", "account_id", "account_tier",
            "region", "valid_from", "valid_to", "is_current",
            "version_number")
    .orderBy("valid_from")
)

# Now demonstrate point-in-time query
# "What tier was this account in as of 2023-01-01?"
query_date = "2023-10-01"

print(f"\nPoint-in-time query: what was the account tier on {query_date}?")
result = df_dim.filter(
    (F.col("account_id") == sample_id) &
    (F.col("valid_from") <= F.lit(query_date)) &
    (
        F.col("valid_to").isNull() |
        (F.col("valid_to") >= F.lit(query_date))
    )
).select("account_id", "account_tier", "region",
         "valid_from", "valid_to", "is_current")

display(result)
print("""
This is what makes SCD2 powerful:
  - The same account_id returns DIFFERENT tier values at different dates
  - Historical reports automatically reflect what was true AT THAT TIME
  - No overwriting of history — every version is preserved permanently
""")

POINT-IN-TIME CORRECTNESS PROOF

Accounts with SCD2 history (sample):


account_id,account_name,versions
ACC01594092CCEF,Jones-Lin,2
ACC05B7DE71153E,Hughes-Alvarado,2
ACC084D36202D5C,Moore Group,2
ACC09E0DE9CFB47,Bailey-Cook,2
ACC0C068320F0B8,"Marshall, Dominguez and Welch",2



Full SCD2 history for account: ACC01594092CCEF


account_sk,account_id,account_tier,region,valid_from,valid_to,is_current,version_number
0,ACC01594092CCEF,Bronze,Europe,2023-09-29,2024-08-31,false,1
1,ACC01594092CCEF,Silver,Europe,2024-09-01,null,true,2



Point-in-time query: what was the account tier on 2023-10-01?


account_id,account_tier,region,valid_from,valid_to,is_current
ACC01594092CCEF,Bronze,Europe,2023-09-29,2024-08-31,false



This is what makes SCD2 powerful:
  - The same account_id returns DIFFERENT tier values at different dates
  - Historical reports automatically reflect what was true AT THAT TIME
  - No overwriting of history — every version is preserved permanently



In [0]:
# ── Final summary ─────────────────────────────────────────────
dims = [
    ("dim_accounts", DIM_ACCOUNTS_PATH),
    ("dim_vessels",  DIM_VESSELS_PATH),
    ("dim_ports",    DIM_PORTS_PATH),
]

print("SCD2 SUMMARY")
print("="*55)
print(f"{'Dimension':<20} {'Total':>8} {'Current':>10} {'Historical':>12}")
print("-"*55)

for name, path in dims:
    df = spark.read.format("delta").load(path)
    total   = df.count()
    current = df.filter(F.col("is_current") == True).count()
    history = total - current
    print(f"{name:<20} {total:>8,} {current:>10,} {history:>12,}")

print("="*55)
print("""
EXPORT THIS NOTEBOOK:
  File → Export → Source File (.py)
  Save to: notebooks/03_silver_scd2.py

PUSH TO GITHUB:
  git add notebooks/03_silver_scd2.py
  git commit -m "Add SCD2 notebook — dim_accounts, dim_vessels, dim_ports"
  git push origin main

NEXT: 04_gold_dimensions — build dim_date, and promote
      silver dims into gold with surrogate key resolution
""")

SCD2 SUMMARY
Dimension               Total    Current   Historical
-------------------------------------------------------
dim_accounts              340        300           40
dim_vessels                48         40            8
dim_ports                  18         15            3

EXPORT THIS NOTEBOOK:
  File → Export → Source File (.py)
  Save to: notebooks/03_silver_scd2.py

PUSH TO GITHUB:
  git add notebooks/03_silver_scd2.py
  git commit -m "Add SCD2 notebook — dim_accounts, dim_vessels, dim_ports"
  git push origin main

NEXT: 04_gold_dimensions — build dim_date, and promote
      silver dims into gold with surrogate key resolution

